# Laya V5.0.1 — All 28 Layers Brute-Force Hybrid SVD Search (Pair-Delta Hotfix)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V50_All28_BruteForce_Hybrid_SVD_Colab.ipynb)

## Based on your V4.9 measured results

The best **passing** V4.9 checkpoint is at **step 120**, with 119/120 teacher agreement (99.17%), 104/120 gold accuracy (86.67%), gold drop 0.83 percentage points and JS 0.000553. V4.9's greedy PROBE selected **7 attention projections** and predicted **35,127,296 bytes** of FP32 unique-weight savings. Later pairs 22–23 and 25–26 were more compression-tolerant than many early pairs. These are PROBE results, not a new AUDIT pass or speed measurement.

## New architecture and all-layer experiment

Load [vtava/Laya-1.2-speedup-standalone](https://huggingface.co/vtava/Laya-1.2-speedup-standalone) with the full 28-layer pretrained ModernBERT backbone and 64-token marker-aware decision head. Test **all 28 encoder blocks**, each with:
- Attention **Wqkv** and **Wo** projection matrices.
- Feed-forward **Wi** and **Wo** projection matrices.
- For nine second local-attention layers, an additional V4.9-style **paired-delta sharing** choice.

Candidate families: **112 independent own-weight SVD** + **18 paired delta-SVD** = **130 families**. At ranks 32, 64, 128 and 256, this is up to **520 single-matrix trials**, before filtering out non-saving settings.

For each candidate, screen on a subset of PROBE, then greedily test combinations on the **whole 120-decision PROBE**. Keep *only* replacements that maintain teacher agreement ≥99%, gold drop ≤1.5pp, and mean JS ≤0.003. Use V4.9's seven successful pairs as *priority hints*, not a quality guarantee.

All other weights remain pretrained and frozen; distill only the selected SVD factors. Checkpoint selection uses PROBE, **one untouched case-disjoint 200-decision AUDIT** is reserved for the end. Paired GPU and synchronized wall-clock speed measure performance against unchanged V4.6.

**Offline standalone export** includes all model weights, tokenizer, config and this module. Conditional private HF destination: `vtava/Laya-v50-all28-bruteforce`. Publication requires **ALL** quality, byte-size, runtime and offline reload gates to pass. No accuracy or speed results are claimed before the notebook runs.

Brute-force trial count can be expensive. Intermediate scans and greedy/training reports are saved after each candidate or checkpoint. The candidate search is exhaustive *over these declared module families and ranks*, not over every theoretically possible architecture.


### V5.0.1 fix and scan resume

The shared-delta `SVDDeltaLinear` call now passes factors with keyword-only `A=` and `B=`; initialized/uninitialized replacement and rollback are checked **before** a long scan. Turn on `RESUME_BRUTEFORCE_SCAN` to reuse `all28_bruteforce_scan.json` from the current Colab runtime. In a new runtime, upload the previously saved JSON and set `RESUME_SCAN_FILE_PATH` to its path. Only completed, source-revision-compatible families are skipped; the greedy search independently measures combinations on the full PROBE set. For older saved scans, recomputed randomized SVD factors can differ from original trial factors, so screen results are heuristic until rechecked.


In [1]:
#@title 1. Pinned requirements, GPU environment and runtime imports
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json,math,time,random,hashlib,copy,statistics,gc,importlib.util
from pathlib import Path
from collections import OrderedDict,Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download,notebook_login
from safetensors.torch import save_file,load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(),"Select a Colab GPU (T4 or better)"
DEVICE=torch.device("cuda")
print("Torch:",torch.__version__,"GPU:",torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 663.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 18.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Torch: 2.11.0+cu130 GPU: Tesla T4


In [2]:
#@title 2. Reproducible configuration, teacher and saved experiment directory
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
STUDENT_REPO="vtava/Laya-v50-all28-bruteforce"
SEED=2047  # same case split as V4.7, V4.8 and V4.9
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=1 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224 #@param {type:"integer"}
TRAIN_STEPS=400 #@param {type:"integer"}
TRAIN_CHECK_EVERY=40 #@param {type:"integer"}
TRAIN_LR=0.00002 #@param {type:"number"}
CANDIDATE_RANKS=(32,64,128,256)
RESUME_BRUTEFORCE_SCAN=True #@param {type:"boolean"}
# Optional full path to a saved all28_bruteforce_scan.json from a previous Colab run.
# If using a NEW Colab session, first upload it into /content and specify its path.
RESUME_SCAN_FILE_PATH="" #@param {type:"string"}
SCREEN_DECISIONS=24 #@param {type:"integer"}
PROBE_DECISIONS=120
AUDIT_DECISIONS=200
MAX_SELECTED=20 #@param {type:"integer"}
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_TEACHER_AGREEMENT=0.99
MAX_GOLD_DROP_PP=1.5
MAX_JS=0.003
MIN_GPU_SPEED_RATIO=0.95
MIN_WALL_SPEED_RATIO=0.95
MIN_SAVED_BYTES=20_000_000
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert 2<=SCREEN_DECISIONS<=PROBE_DECISIONS
assert 1<=TRAIN_B<=2 and 1<=BATCH<=8 and TRAIN_MAX_TOKENS>=192
assert TRAIN_STEPS>=0 and TRAIN_CHECK_EVERY>=1 and BENCH_REPEATS>=5
assert MAX_SELECTED>=1 and len(CANDIDATE_RANKS)>=1
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
device=DEVICE  # V4.6 dataset benchmark helper expects this name
OUT=Path("/content/Laya_V50_All28_BruteForce")
OUT.mkdir(parents=True,exist_ok=True)
DEST=OUT/"standalone_model"
DEST.mkdir(parents=True,exist_ok=True)
api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO)
except Exception as err:
    print("Source HF model may be private; requesting login:",repr(err))
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO)
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=teacher_info.sha))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert int(source_config["head_context_budget"])==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
assert tokenizer.pad_token_id is not None
print("V4.6 teacher:",TEACHER_REPO,teacher_info.sha)
print("Reference model.safetensors bytes:",(parent_dir/"model.safetensors").stat().st_size)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

requirements.txt:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

V4.6 teacher: vtava/Laya-1.2-speedup-standalone b60184b2976e9f7ffa60c1c42b221d81b3691c3d
Reference model.safetensors bytes: 1684141004


In [3]:
#@title 3. Embed fully offline V5.0 runtime and load TWO independent complete models
V50_RUNTIME_CODE = "\"\"\"Laya V5.0 all-28-layer brute-force hybrid SVD compression, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\n\n# V4.9 does NOT force compression of all nine local pairs.\n# Each selected follower projection gets a separate low-rank residual;\n# all unselected attention projection matrices retain pretrained weights.\ndef projection_slots(model):\n    slots=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            a=getattr(model.encoder.layers[first].attn,component)\n            b=getattr(model.encoder.layers[second].attn,component)\n            if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n                raise TypeError(f\"Expected nn.Linear for layer {second}.{component}\")\n            if a.weight.shape != b.weight.shape:\n                raise ValueError(f\"Incompatible original projection: {second}.{component}\")\n            if a.bias is not None or b.bias is not None:\n                raise ValueError(f\"Biasful projection needs separate residual bias: {second}.{component}\")\n            slots.append((first,second,component))\n    return slots\n\n\ndef _validate_slot(model, first, second, component):\n    \"\"\"Validate schedule independently of other already-compressed slots.\"\"\"\n    first,second=int(first),int(second)\n    component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Unknown local pair: {(first,second)}\")\n    if component not in (\"Wqkv\",\"Wo\"):\n        raise ValueError(f\"Unknown projection: {component}\")\n    return first,second,component\n\n\ndef svd_factors(model, first, second, component, max_rank=256,\n                oversampling=8, niter=2):\n    \"\"\"Compute one factorization per projection from ORIGINAL pretrained weights.\n\n    A and B are float32 CPU tensors and can be sliced for different ranks.\n    \"\"\"\n    first,second,component=_validate_slot(model,first,second,component)\n    a=getattr(model.encoder.layers[first].attn,component)\n    b=getattr(model.encoder.layers[second].attn,component)\n    if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n        raise ValueError(\"SVD must be done BEFORE that follower is compressed\")\n    with torch.no_grad():\n        delta=b.weight.detach().float()-a.weight.detach().float()\n        r=min(int(max_rank),min(delta.shape)-1)\n        q=min(r+int(oversampling),min(delta.shape))\n        # Randomized SVD (approximation, not an exact mathematical SVD).\n        U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n        root=S[:r].clamp_min(0).sqrt()\n        A=(U[:,:r]*root.unsqueeze(0)).cpu().contiguous()\n        B=(root.unsqueeze(1)*V[:,:r].T).cpu().contiguous()\n        total=float(delta.square().sum().item())\n        error=float((delta-(A.to(delta.device)@B.to(delta.device))).square().sum().item())\n        del U,S,V,delta\n    return A,B,{\"rank\":r,\n        \"relative_frobenius_error\":(error/max(total,1e-30))**0.5,\n        \"saved_parameter_bytes_if_rank256\":None}\n\n\ndef build_adapter(model, first, second, component, rank, factors=None,\n                  *, initialized=True):\n    \"\"\"Replace ONE follower's projection; return original module for rollback.\n\n    If initialized=False, A/B are zeros until strictly loaded from checkpoint.\n    \"\"\"\n    first=int(first);second=int(second);component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Invalid local pair: {(first,second)}\")\n    leader=getattr(model.encoder.layers[first].attn,component)\n    follower_block=model.encoder.layers[second].attn\n    old=getattr(follower_block,component)\n    if not isinstance(leader,nn.Linear) or not isinstance(old,nn.Linear):\n        raise TypeError(\"Projection already wrapped or unavailable\")\n    if leader.bias is not None or old.bias is not None:\n        raise ValueError(\"Biasful attention projections are unsupported\")\n    if initialized:\n        if factors is None:\n            A,B,_=svd_factors(model,first,second,component,max_rank=rank)\n        else:\n            A,B=factors\n        adapter=SVDDeltaLinear(leader,rank,A=A[:,:rank],B=B[:rank,:])\n    else:\n        adapter=SVDDeltaLinear(leader,rank)\n    setattr(follower_block,component,adapter)\n    return old\n\n\ndef restore_original_projection(model, second, component, original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Rollback needs the original pretrained linear module\")\n    setattr(model.encoder.layers[int(second)].attn,str(component),original)\n\n\ndef selected_adapters(model):\n    selected=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            layer=getattr(model.encoder.layers[second].attn,component)\n            if isinstance(layer,SVDDeltaLinear):\n                selected.append((first,second,component,layer))\n    return selected\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef estimated_saved_bytes(model, first, second, component, rank):\n    leader=getattr(model.encoder.layers[first].attn,component)\n    outdim,indim=leader.weight.shape\n    # Full follower weight removed; A (out×rank), B (rank×in) remain.\n    return 4*(outdim*indim-int(rank)*(outdim+indim))\n\n\ndef load_v49_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Strict, fully offline restoration of selectively compressed model.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if meta[\"format\"]!=\"laya-v49-selective-svd-v1\":\n        raise ValueError(\"Wrong model format: expected V4.9 selective SVD\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    for item in meta[\"compressed_projections\"]:\n        build_adapter(model,int(item[\"first\"]),int(item[\"second\"]),\n                      str(item[\"component\"]),int(item[\"rank\"]),\n                      initialized=False)\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    # No duplicated original follower weights or duplicated leader submodule.\n    assert len(selected_adapters(model))==len(meta[\"compressed_projections\"])\n    for first,second,component,wrapper in selected_adapters(model):\n        leader=getattr(model.encoder.layers[first].attn,component)\n        assert wrapper._shared is leader\n        assert \"_shared\" not in wrapper._modules\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ---------------------------------------------------------------------------\n# V5.0: brute-force all 28 ModernBERT blocks, attention + FFN, preserving the\n# proven V4.9 pretrained-pair DELTA path as one selectable candidate family.\n# ---------------------------------------------------------------------------\n\nclass IndependentSVDLinear(nn.Module):\n    \"\"\"Plain original projection compressed as A @ B; original dense W omitted.\n\n    Compatible with ModernBERT's attention Wqkv, Wo and MLP Wi, Wo calls.\n    A saved bias is kept if the original linear was biased.\n    \"\"\"\n    def __init__(self, original: nn.Linear, rank: int, A=None, B=None):\n        super().__init__()\n        if not isinstance(original, nn.Linear):\n            raise TypeError(\"Expected original nn.Linear\")\n        out_dim,in_dim=original.weight.shape\n        rank=int(rank)\n        if rank<=0 or rank>=min(out_dim,in_dim):\n            raise ValueError(f\"Bad SVD rank {rank} for {out_dim}x{in_dim}\")\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),\n            dtype=original.weight.dtype,device=original.weight.device))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),\n            dtype=original.weight.dtype,device=original.weight.device))\n        if original.bias is None:\n            self.register_parameter(\"bias\",None)\n        else:\n            self.bias=nn.Parameter(original.bias.detach().clone())\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A[:,:rank].to(device=self.A.device,dtype=self.A.dtype))\n                self.B.copy_(B[:rank,:].to(device=self.B.device,dtype=self.B.dtype))\n\n    def forward(self,x):\n        return F.linear(F.linear(x,self.B),self.A,self.bias)\n\n\ndef all_encoder_slots(model):\n    \"\"\"Enumerate available projections in EVERY pretrained encoder layer.\"\"\"\n    slots=[]\n    for index,layer in enumerate(model.encoder.layers):\n        for block,names in ((\"attn\",(\"Wqkv\",\"Wo\")),(\"mlp\",(\"Wi\",\"Wo\"))):\n            component=getattr(layer,block)\n            for name in names:\n                module=getattr(component,name,None)\n                if module is None:\n                    raise ValueError(f\"Required projection layer {index}.{block}.{name} missing\")\n                if not isinstance(module,nn.Linear):\n                    raise TypeError(f\"Not an uncompressed nn.Linear: {index}.{block}.{name}: {type(module)}\")\n                slots.append((index,block,name))\n    if len(slots)!=4*len(model.encoder.layers):\n        raise RuntimeError(\"Expected exactly two attention and two FFN projections per layer\")\n    return slots\n\n\ndef list_candidate_families(model):\n    \"\"\"Full brute-force search space: 112 own-SVD + 18 delta-shared slots.\"\"\"\n    options=[{\"layer\":int(layer),\"block\":block,\"component\":name,\n              \"mode\":\"own_svd\"}\n             for layer,block,name in all_encoder_slots(model)]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            options.append({\"layer\":int(second),\"leader\":int(first),\n                            \"block\":\"attn\",\"component\":component,\n                            \"mode\":\"pair_delta\"})\n    return options\n\n\ndef own_svd_factors(original, *, max_rank=256, oversampling=8,niter=2):\n    \"\"\"Randomized, truncated SVD of ORIGINAL pretrained W (not layer delta).\"\"\"\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Expected an uncompressed linear projection\")\n    with torch.no_grad():\n        weight=original.weight.detach().float()\n        rank=min(int(max_rank),min(weight.shape)-1)\n        q=min(rank+int(oversampling),min(weight.shape))\n        U,S,V=torch.svd_lowrank(weight,q=q,niter=int(niter))\n        scale=S[:rank].clamp_min(0).sqrt()\n        A=(U[:,:rank]*scale[None,:]).cpu().contiguous()\n        B=(scale[:,None]*V[:,:rank].T).cpu().contiguous()\n        approximated=A.to(weight.device)@B.to(weight.device)\n        numerator=float((weight-approximated).square().sum().item())\n        denominator=float(weight.square().sum().item())\n    return A,B,{\"rank\":rank,\n                \"relative_frobenius_error\":(numerator/max(denominator,1e-30))**.5}\n\n\ndef projection_at(model, spec):\n    return getattr(getattr(model.encoder.layers[int(spec[\"layer\"])],\n                           str(spec[\"block\"])),str(spec[\"component\"]))\n\n\ndef replace_projection(model, spec, *, factors=None, initialized=True):\n    \"\"\"Apply one projection replacement; return original module for rollback.\"\"\"\n    idx=int(spec[\"layer\"])\n    block=str(spec[\"block\"])\n    comp=str(spec[\"component\"])\n    rank=int(spec[\"rank\"])\n    mode=str(spec[\"mode\"])\n    owner=getattr(model.encoder.layers[idx],block)\n    old=getattr(owner,comp)\n    if not isinstance(old,nn.Linear):\n        raise TypeError(f\"Already compressed or unsupported {idx}.{block}.{comp}\")\n    if mode==\"own_svd\":\n        if initialized and factors is None:\n            factors=own_svd_factors(old,max_rank=rank)[:2]\n        if initialized:\n            A,B=factors\n            replacement=IndependentSVDLinear(old,rank,A,B)\n        else:\n            replacement=IndependentSVDLinear(old,rank)\n    elif mode==\"pair_delta\":\n        if block!=\"attn\" or comp not in (\"Wqkv\",\"Wo\"):\n            raise ValueError(\"Pair delta only supported for local attention\")\n        leader=int(spec[\"leader\"])\n        if (leader,idx) not in local_pair_indices(model):\n            raise ValueError(f\"Not a valid local pair: {leader},{idx}\")\n        shared=getattr(model.encoder.layers[leader].attn,comp)\n        if not isinstance(shared,nn.Linear):\n            raise TypeError(\"Leader must retain original pretrained linear\")\n        if shared.bias is not None or old.bias is not None:\n            raise ValueError(\"Pair delta only supported for bias-free attention\")\n        if initialized:\n            if factors is None:\n                A,B,_=svd_factors(model,leader,idx,comp,max_rank=rank)\n            else:\n                A,B=factors\n            replacement=SVDDeltaLinear(shared,rank,A=A[:,:rank],B=B[:rank,:])\n        else:\n            replacement=SVDDeltaLinear(shared,rank)\n    else:\n        raise ValueError(f\"Unsupported family: {mode}\")\n    setattr(owner,comp,replacement)\n    return old\n\n\ndef revert_projection(model,spec,original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Original projection must be nn.Linear\")\n    owner=getattr(model.encoder.layers[int(spec[\"layer\"])],str(spec[\"block\"]))\n    setattr(owner,str(spec[\"component\"]),original)\n\n\ndef applied_adapters(model,selected_specs):\n    result=[]\n    for spec in selected_specs:\n        wrapper=projection_at(model,spec)\n        if not isinstance(wrapper,(IndependentSVDLinear,SVDDeltaLinear)):\n            raise AssertionError(f\"Missing compressed adapter for {spec}\")\n        result.append(wrapper)\n    return result\n\n\ndef expected_saved_bytes(model,spec):\n    \"\"\"FP32 parameter saving for one replacement, including any old bias.\"\"\"\n    idx=int(spec[\"layer\"])\n    owner=getattr(model.encoder.layers[idx],str(spec[\"block\"]))\n    original=getattr(owner,str(spec[\"component\"]))\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Call before installing candidate\")\n    out_dim,in_dim=original.weight.shape\n    rank=int(spec[\"rank\"])\n    # Bias exists in both old and new for own_svd; cancels out.\n    # Pair delta has no bias.\n    return int(4*(out_dim*in_dim-rank*(out_dim+in_dim)))\n\n\ndef load_v50_model(model_dir, *, device=\"cpu\",compile_head=False):\n    \"\"\"Strict reload of all hybrid low-rank projections from local disk only.\"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info[\"format\"]!=\"laya-v50-all28-bruteforce-v1\":\n        raise ValueError(\"Not a V5.0 hybrid model\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    # Rebuild in the same order: only uncompressed leaders are valid for delta.\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    for spec,adapter in zip(info[\"compressed_projections\"],\n                            applied_adapters(model,info[\"compressed_projections\"])):\n        if spec[\"mode\"]==\"pair_delta\":\n            leader=getattr(model.encoder.layers[int(spec[\"leader\"])].attn,\n                           str(spec[\"component\"]))\n            assert adapter._shared is leader\n            assert \"_shared\" not in adapter._modules\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tok\n"
runtime_file=OUT/"modeling_laya_v50.py"
runtime_file.write_text(V50_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("v50_runtime",str(runtime_file))
assert spec and spec.loader
v50=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v50)
normalize_question=v50.normalize_question
build_sequence=v50.build_sequence
collate_items=v50.collate_items
QTYPES=v50.QTYPES
assert all(callable(x) for x in (normalize_question,build_sequence,collate_items))
teacher,_=v50.load_model(parent_dir,device="cuda")
student,_=v50.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v50.unique_parameter_count(teacher)
slots=v50.all_encoder_slots(student)
families=v50.list_candidate_families(student)
assert len(student.encoder.layers)==28
assert len(slots)==112 and len(families)==130
assert len(v50.local_pair_indices(student))==9
assert len(set(tuple(x.items()) for x in families))==130
print("Encoder layers:",len(student.encoder.layers))
print("Original parameters:",f"{native_unique:,}")
print("Brute-force families:",len(families),"of which",len(slots),"are own-weight SVD")
print("Expected individual rank tests:",len(families)*len(CANDIDATE_RANKS))
# Early architecture smoke tests catch incorrect SVD matrix orientations
# and bias handling before the long brute-force GPU search.
with torch.no_grad():
    toy=nn.Linear(16,24,bias=True).to(DEVICE)
    Atoy,Btoy,_=v50.own_svd_factors(toy,max_rank=8)
    compressed_toy=v50.IndependentSVDLinear(toy,8,Atoy,Btoy)
    inputs=torch.randn(3,5,16,device=DEVICE)
    toy_output=compressed_toy(inputs)
    assert toy_output.shape==(3,5,24)
    assert torch.isfinite(toy_output).all()
    del toy,compressed_toy,inputs,toy_output,Atoy,Btoy
gc.collect()
print("✓ Own-weight low-rank projection smoke test passed")

# Preflight V5-specific paired-delta family BEFORE 500+ expensive SVD trials.
# Historical bug: SVDDeltaLinear has keyword-only A/B but replace_projection()
# passed both positionally, failing at the very first pair_delta family.
first_idx, second_idx = v50.local_pair_indices(student)[0]
delta_spec=dict(layer=int(second_idx), leader=int(first_idx), block="attn",
                component="Wo", mode="pair_delta", rank=8)
follower=v50.projection_at(student,delta_spec)
leader=student.encoder.layers[first_idx].attn.Wo
height,width=follower.weight.shape
toyA=torch.randn(height,8,device=DEVICE,dtype=leader.weight.dtype)*0.005
toyB=torch.randn(8,width,device=DEVICE,dtype=leader.weight.dtype)*0.005
test_x=torch.randn(2,3,width,device=DEVICE,dtype=leader.weight.dtype)
old_delta=v50.replace_projection(student,delta_spec,
                                 factors=(toyA,toyB),initialized=True)
try:
    adapter=v50.projection_at(student,delta_spec)
    assert isinstance(adapter,v50.SVDDeltaLinear)
    assert adapter._shared is leader
    assert "_shared" not in adapter._modules, "Shared leader accidentally serialized twice"
    with torch.inference_mode():
        expected=F.linear(test_x,leader.weight,leader.bias)+F.linear(
            F.linear(test_x,toyB),toyA)
        actual=adapter(test_x)
    torch.testing.assert_close(actual,expected,atol=1e-4,rtol=1e-4)
finally:
    v50.revert_projection(student,delta_spec,old_delta)
assert v50.projection_at(student,delta_spec) is follower
# Reload constructor path: instantiated without factor initialization, then
# load_state_dict(strict=True) will restore A/B from the saved checkpoint.
old_uninitialized=v50.replace_projection(student,delta_spec,initialized=False)
try:
    fresh=v50.projection_at(student,delta_spec)
    assert torch.count_nonzero(fresh.A)==0
    assert torch.count_nonzero(fresh.B)==0
finally:
    v50.revert_projection(student,delta_spec,old_uninitialized)
assert v50.projection_at(student,delta_spec) is follower
del toyA,toyB,test_x,old_delta,old_uninitialized,expected,actual
gc.collect()
print("✓ Pair-delta initialized, rollback, keyword-only factors and strict-reload constructors passed")


Encoder layers: 28
Original parameters: 421,029,889
Brute-force families: 130 of which 112 are own-weight SVD
Expected individual rank tests: 520
✓ Own-weight low-rank projection smoke test passed
✓ Pair-delta initialized, rollback, keyword-only factors and strict-reload constructors passed


In [4]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

assert len(probe_items)>=PROBE_DECISIONS and len(audit_items)>=AUDIT_DECISIONS
assert {x["case_id"] for x in probe_items}.isdisjoint({x["case_id"] for x in audit_items})
print("TRAIN/PROBE/AUDIT items:",len(train_items),len(probe_items),len(audit_items))


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

TRAIN/PROBE/AUDIT items: 2500 600 600


In [5]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO)



In [6]:
#@title 6. Strict PROBE quality references and candidate helpers
PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
reference_quality=quality(student,probe_items,PROBE_REF)
assert reference_quality["teacher_agreement"]==1.0
assert reference_quality["mean_js"]<1e-7,reference_quality

def quality_gate(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100. and q["mean_js"]<=MAX_JS)

SCREEN_ITEMS=probe_items[:SCREEN_DECISIONS]
SCREEN_REF=PROBE_REF[:SCREEN_DECISIONS]
# Lightweight screening only: still run mandatory full PROBE before ANY
# combined projection is permanently accepted.
def screen_gate(q):
    allowed=max(1,int(SCREEN_DECISIONS*.10))
    return (q["teacher_agreement"]>=1-allowed/SCREEN_DECISIONS and
            q["gold_drop"]<=0.10 and q["mean_js"]<=0.008)

# V4.9 confirmed these seven *sequential combinations* on 120 PROBE cases.
# They receive priority in the greedy search, but can still be rejected.
V49_PRIORITY=[
    (26,"attn","Wqkv","pair_delta",64),
    (23,"attn","Wqkv","pair_delta",256),
    (26,"attn","Wo","pair_delta",64),
    (23,"attn","Wo","pair_delta",64),
    (8,"attn","Wo","pair_delta",64),
    (17,"attn","Wo","pair_delta",256),
    (20,"attn","Wo","pair_delta",256),
]
print("Baseline PROBE:",reference_quality)
print("Prior successful V4.9 combinations:",len(V49_PRIORITY))


Baseline PROBE: {'decisions': 120, 'teacher_agreement': 1.0, 'gold_accuracy': 0.875, 'teacher_gold_accuracy': 0.875, 'gold_drop': 0.0, 'mean_js': 0.0}
Prior successful V4.9 combinations: 7


In [7]:
#@title 7. BRUTE-FORCE ALL 28 LAYERS — 112 own-SVD + 18 shared-delta families
# Each of the 130 linear-projection families is tested at all candidate ranks
# that save actual storage. All models start from the exact same pretrained
# student; temporary replacements are rolled back after each small PROBE.
#
# The original 28-layer model stays loaded, no additional model copies are
# created for each candidate. SVD factors are reused across rank alternatives.
cache={}
screen_log=[]
family_log=[]
total_trials=0
start_scan=time.perf_counter()

def family_key(f):
    return (int(f["layer"]),str(f["block"]),str(f["component"]),str(f["mode"]))

# Resume only complete families from a validated saved screen-log prefix.
scan_file=OUT/"all28_bruteforce_scan.json"
if RESUME_SCAN_FILE_PATH.strip():
    external=Path(RESUME_SCAN_FILE_PATH.strip()).expanduser()
    if not external.is_file():
        raise FileNotFoundError(f"RESUME_SCAN_FILE_PATH not found: {external}")
    if external.resolve()!=scan_file.resolve():
        import shutil
        shutil.copy2(external,scan_file)
completed_keys=set()
if RESUME_BRUTEFORCE_SCAN and scan_file.is_file():
    previous=json.loads(scan_file.read_text(encoding="utf-8"))
    saved_families=previous.get("families",[])
    valid=(
        previous.get("model")==TEACHER_REPO and
        previous.get("teacher_revision")==teacher_info.sha and
        previous.get("total_families")==len(families) and
        previous.get("screen_decisions")==SCREEN_DECISIONS and
        all(log.get("family")==families[i] for i,log in enumerate(saved_families)
            if i<len(families)) and len(saved_families)<=len(families))
    if valid and previous.get("candidate_ranks") is not None:
        valid=tuple(previous["candidate_ranks"])==tuple(CANDIDATE_RANKS)
    if valid:
        expected=[]
        for fam in saved_families:
            specbase=fam["family"]
            for candidate_rank in CANDIDATE_RANKS:
                old_spec=dict(specbase,rank=int(candidate_rank))
                # Compute storage without modifying the pretrained student.
                if v50.expected_saved_bytes(student,old_spec)>0:
                    expected.append(old_spec)
        existing=previous.get("trials",[])
        valid=(len(existing)==len(expected) and
               all(entry.get("spec")==spec for entry,spec in zip(existing,expected)))
    if valid:
        family_log=list(saved_families)
        screen_log=list(previous["trials"])
        total_trials=len(screen_log)
        completed_keys={family_key(x["family"]) for x in family_log}
        print(f"RESUMED {len(completed_keys)}/{len(families)} complete "
              f"families and {total_trials} rank trials from",scan_file)
        if not previous.get("deterministic_factors"):
            print("Legacy scan: re-factored SVD candidates may differ. "
                  "Each choice is still independently re-evaluated on full PROBE.")
    else:
        print("Saved scan mismatches model/ranks/family prefix; starting fresh.")
        # Retain invalid file for diagnosis, never mix two incompatible scans.
        scan_file.rename(scan_file.with_suffix(".incompatible.json"))

def compute_factors(family):
    """Deterministic family-specific SVD for new screens and lazy resume."""
    key=family_key(family)
    if key in cache:
        return cache[key]
    seed_label=":".join(map(str,key))
    family_seed=SEED+sum((i+1)*ord(ch) for i,ch in enumerate(seed_label))
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(family_seed)
        if family["mode"]=="own_svd":
            A,B,diag=v50.own_svd_factors(
                v50.projection_at(student,family),
                max_rank=max(CANDIDATE_RANKS),oversampling=8,niter=2)
        else:
            A,B,diag=v50.svd_factors(
                student,int(family["leader"]),int(family["layer"]),
                str(family["component"]),max_rank=max(CANDIDATE_RANKS),
                oversampling=8,niter=2)
    cache[key]=(A,B)
    return A,B,diag

for family_idx,family in enumerate(families,1):
    if family_key(family) in completed_keys:
        continue
    layer=int(family["layer"])
    block=family["block"]
    component=family["component"]
    mode=family["mode"]
    key=family_key(family)
    assert isinstance(v50.projection_at(student,family),nn.Linear),key
    A,B,diag=compute_factors(family)
    family_log.append(dict(family=family,svd_diagnostics=diag))

    for rank in CANDIDATE_RANKS:
        spec=dict(family,rank=int(rank))
        storage_saved=v50.expected_saved_bytes(student,spec)
        if storage_saved<=0:
            continue
        original=v50.replace_projection(
            student,spec,factors=(A,B),initialized=True)
        try:
            q=quality(student,SCREEN_ITEMS,SCREEN_REF)
        finally:
            v50.revert_projection(student,spec,original)
        rec=dict(spec=spec,screen_quality=q,screen_pass=screen_gate(q),
                 estimated_saved_bytes=storage_saved)
        screen_log.append(rec)
        total_trials+=1
        if total_trials<=12 or total_trials%20==0:
            print(f"TRIAL {total_trials} layer={layer:02d} {block}.{component} "
                  f"mode={mode} rank={rank}: agree={q['teacher_agreement']:.3f} "
                  f"JS={q['mean_js']:.6f} save={storage_saved/1e6:.2f}MB "
                  f"screen={rec['screen_pass']}")
    (OUT/"all28_bruteforce_scan.json").write_text(
        json.dumps(dict(model=TEACHER_REPO,teacher_revision=teacher_info.sha,
                        total_families=len(families),
                        completed_families=family_idx,
                        screen_decisions=SCREEN_DECISIONS,
                        candidate_ranks=list(CANDIDATE_RANKS),
                        deterministic_factors=True,
                        trials=screen_log,
                        families=family_log),indent=2))
assert len(screen_log)>=len(families)
assert len(v50.applied_adapters(student,[]))==0
print("Completed all",len(families),"families with",total_trials,"rank tests")
print("Pre-screen survivors:",sum(r["screen_pass"] for r in screen_log))
print("Scan elapsed minutes:",round((time.perf_counter()-start_scan)/60,2))


TRIAL 1 layer=00 attn.Wqkv mode=own_svd rank=32: agree=0.708 JS=0.009023 save=12.06MB screen=False
TRIAL 2 layer=00 attn.Wqkv mode=own_svd rank=64: agree=0.875 JS=0.005208 save=11.53MB screen=False
TRIAL 3 layer=00 attn.Wqkv mode=own_svd rank=128: agree=0.875 JS=0.001830 save=10.49MB screen=False
TRIAL 4 layer=00 attn.Wqkv mode=own_svd rank=256: agree=0.917 JS=0.000903 save=8.39MB screen=True
TRIAL 5 layer=00 attn.Wo mode=own_svd rank=32: agree=0.875 JS=0.002553 save=3.93MB screen=False
TRIAL 6 layer=00 attn.Wo mode=own_svd rank=64: agree=0.958 JS=0.001677 save=3.67MB screen=True
TRIAL 7 layer=00 attn.Wo mode=own_svd rank=128: agree=1.000 JS=0.001268 save=3.15MB screen=True
TRIAL 8 layer=00 attn.Wo mode=own_svd rank=256: agree=0.958 JS=0.000433 save=2.10MB screen=True
TRIAL 9 layer=00 mlp.Wi mode=own_svd rank=32: agree=0.875 JS=0.005803 save=20.69MB screen=False
TRIAL 10 layer=00 mlp.Wi mode=own_svd rank=64: agree=0.875 JS=0.003561 save=19.89MB screen=False
TRIAL 11 layer=00 mlp.Wi mod

In [11]:
#@title 8. Full-PROBE greedy combination search with conflict detection and rollback
# Every accepted candidate has passed full case-disjoint PROBE *in combination*
# with all previously selected replacements. We do not pretend that the
# 24-decision screen proves quality. V4.9's passing sequence gets priority.
def ordering(entry):
    s=entry["spec"]
    k=(int(s["layer"]),s["block"],s["component"],s["mode"],int(s["rank"]))
    if k in V49_PRIORITY:
        return (0,V49_PRIORITY.index(k),0,0)
    # Other candidates: prioritize high savings that minimally distort
    # teacher scores, rather than indiscriminate high compression.
    q=entry["screen_quality"]
    return (1, -float(entry["estimated_saved_bytes"])/
            max(q["mean_js"],1e-5), -q["teacher_agreement"],
            -entry["estimated_saved_bytes"])

candidates=sorted((r for r in screen_log if r["screen_pass"]),key=ordering)
selected_specs=[]
selection_attempts=[]
occupied=set()

def is_conflict(spec):
    slot=(spec["layer"],spec["block"],spec["component"])
    if slot in occupied:return True
    if spec["mode"]=="pair_delta":
        leader_slot=(spec["leader"],"attn",spec["component"])
        if leader_slot in occupied:return True
    for selected in selected_specs:
        if (selected["mode"]=="pair_delta" and
            (spec["layer"],spec["block"],spec["component"])==
            (selected["leader"],"attn",selected["component"])):
            return True
    return False

for entry in candidates:
    if len(selected_specs)>=MAX_SELECTED:break
    spec=entry["spec"]
    if is_conflict(spec):continue
    key=family_key(spec)
    # Rebuild factors lazily if this family was restored from a prior scan.
    A,B=compute_factors(spec)
    old=v50.replace_projection(student,spec,factors=(A,B))
    trial=quality(student,probe_items,PROBE_REF)
    passed=quality_gate(trial)
    row=dict(spec=spec,quality=trial,accepted=passed,
             estimated_saved_bytes=entry["estimated_saved_bytes"])
    selection_attempts.append(row)
    if passed:
        selected_specs.append(spec)
        occupied.add((spec["layer"],spec["block"],spec["component"]))
    else:
        v50.revert_projection(student,spec,old)
    print("COMBINATION",len(selection_attempts),"slot",spec["layer"],
          spec["block"],spec["component"],spec["mode"],spec["rank"],
          "accepted",passed,"JS",round(trial["mean_js"],7),
          "agreement",round(trial["teacher_agreement"],4))
    (OUT/"greedy_combinations.json").write_text(
        json.dumps(dict(selected=selected_specs,attempts=selection_attempts),indent=2))

selection_mode="full_PROBE_gate"
if not selected_specs:
    # A genuinely empty validated search is a result, but for experimental
    # recovery allow the single least-distorting candidate to be distilled.
    # It is NOT validated and will never be published unless final gates pass.
    fallback=min(screen_log,key=lambda x:x["screen_quality"]["mean_js"])
    spec=fallback["spec"]
    A,B=compute_factors(spec)
    v50.replace_projection(student,spec,factors=(A,B))
    selected_specs=[spec]
    selection_mode="one_unvalidated_candidate_for_distillation"
    (OUT/"greedy_combinations.json").write_text(
        json.dumps(dict(selected=selected_specs,attempts=selection_attempts,
                        mode=selection_mode),indent=2))
unique_student=v50.unique_parameter_count(student)
predicted_saving_bytes=(native_unique-unique_student)*4
assert predicted_saving_bytes>0,(native_unique,unique_student)

# These cached factors are not necessary for training or export.
del cache
gc.collect()


COMBINATION 1 slot 26 attn Wqkv pair_delta 64 accepted True JS 1.52e-05 agreement 1.0
COMBINATION 2 slot 23 attn Wqkv pair_delta 256 accepted False JS 4.4e-05 agreement 0.9833
COMBINATION 3 slot 26 attn Wo pair_delta 64 accepted True JS 1.57e-05 agreement 0.9917
COMBINATION 4 slot 23 attn Wo pair_delta 64 accepted False JS 3.19e-05 agreement 0.9833
COMBINATION 5 slot 8 attn Wo pair_delta 64 accepted False JS 0.0004955 agreement 0.9833
COMBINATION 6 slot 17 attn Wo pair_delta 256 accepted True JS 8.12e-05 agreement 0.9917
COMBINATION 7 slot 20 attn Wo pair_delta 256 accepted True JS 0.0001116 agreement 0.9917
COMBINATION 8 slot 27 attn Wqkv own_svd 64 accepted True JS 0.0001012 agreement 0.9917
COMBINATION 9 slot 24 mlp Wi own_svd 256 accepted True JS 0.0001494 agreement 0.9917
COMBINATION 10 slot 23 attn Wqkv own_svd 256 accepted True JS 0.000153 agreement 0.9917
COMBINATION 11 slot 24 mlp Wo own_svd 256 accepted False JS 0.0001522 agreement 0.9833
COMBINATION 12 slot 24 mlp Wo own_svd

8

In [12]:
#@title 9. Distill selected SVD adapters only; keep everything else frozen
student.requires_grad_(False)
adapted=v50.applied_adapters(student,selected_specs)
trainable=[]
for module in adapted:
    trainable.extend((module.A,module.B))
for p in trainable:p.requires_grad_(True)
assert len(trainable)==2*len(selected_specs)
assert sum(p.numel() for p in student.parameters() if p.requires_grad)==sum(
    p.numel() for p in trainable)
student.eval()
optimizer=torch.optim.AdamW(trainable,lr=TRAIN_LR,weight_decay=0.)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)

def save_trainable():
    return [(m.A.detach().cpu().clone(),m.B.detach().cpu().clone())
            for m in v50.applied_adapters(student,selected_specs)]
def restore_trainable(state):
    with torch.no_grad():
        for m,(A,B) in zip(v50.applied_adapters(student,selected_specs),state):
            m.A.copy_(A.to(device=m.A.device,dtype=m.A.dtype))
            m.B.copy_(B.to(device=m.B.device,dtype=m.B.dtype))

best_step=0
best_state=save_trainable()
best_quality=quality(student,probe_items,PROBE_REF)
best_pass=quality_gate(best_quality)
training_log=[]

for step in range(1,TRAIN_STEPS+1):
    b=batch(train_items,((step-1)*TRAIN_B)%len(train_items),TRAIN_B)
    assert bool((b["marker_pos"][b["marker_mask"]]<TRAIN_MAX_TOKENS).all()),(
        "Truncation would remove an option marker: increase TRAIN_MAX_TOKENS")
    for key in ("input_ids","attention_mask"):
        b[key]=b[key][:,:TRAIN_MAX_TOKENS]
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**b)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        student_logits,_=student(**b)
        tau=2.0
        target=F.softmax(teacher_logits.float()/tau,dim=-1)
        log_prob=F.log_softmax(student_logits.float()/tau,dim=-1)
        loss=F.kl_div(log_prob,target,reduction="batchmean")*(tau**2)
    assert bool(torch.isfinite(loss)),("Nonfinite loss",step)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable,1.0)
    scaler.step(optimizer)
    scaler.update()
    if step%TRAIN_CHECK_EVERY==0 or step==TRAIN_STEPS:
        scored=quality(student,probe_items,PROBE_REF)
        passed=quality_gate(scored)
        improved=(passed and not best_pass) or (
            passed==best_pass and scored["mean_js"]<best_quality["mean_js"])
        if improved:
            best_state=save_trainable()
            best_quality=scored
            best_pass=passed
            best_step=step
        item=dict(step=step,loss=float(loss.detach().cpu()),
                  quality=scored,passed=passed,selected=improved)
        training_log.append(item)
        (OUT/"training_progress.json").write_text(
            json.dumps(training_log,indent=2))
        print("STEP",step,"loss",item["loss"],
              "agree",scored["teacher_agreement"],
              "gold",scored["gold_accuracy"],"JS",scored["mean_js"],
              "pass",passed,"selected",improved)
restore_trainable(best_state)
student.eval().requires_grad_(False)
print("FROZEN checkpoint: selected step",best_step,"probe",best_quality,
      "passed",best_pass)


STEP 40 loss 0.0027006864547729492 agree 0.9916666666666667 gold 0.8833333333333333 JS 0.001151722076247097 pass True selected True
STEP 80 loss 0.02266073226928711 agree 0.9916666666666667 gold 0.8833333333333333 JS 0.001186026976559636 pass True selected False
STEP 120 loss 0.00025141239166259766 agree 1.0 gold 0.875 JS 0.001059634604916937 pass True selected True
STEP 160 loss 1.1920928955078125e-07 agree 1.0 gold 0.875 JS 0.0009203133611663361 pass True selected True
STEP 200 loss 0.0001010894775390625 agree 1.0 gold 0.875 JS 0.0008820693410598324 pass True selected True
STEP 240 loss 0.0011193156242370605 agree 0.9916666666666667 gold 0.8666666666666667 JS 0.000756681559929954 pass True selected True
STEP 280 loss 0.000988602638244629 agree 0.975 gold 0.8833333333333333 JS 0.0008145688388746446 pass False selected False
STEP 320 loss 0.0015047788619995117 agree 0.9833333333333333 gold 0.875 JS 0.0008813947092069914 pass False selected False
STEP 360 loss 0.020736098289489746 agree

In [13]:
#@title 10. Frozen PROBE speed and ONE untouched 200-decision AUDIT
final_probe=quality(student,probe_items,PROBE_REF)
probe_perf=benchmark_paired(teacher,student,repeats=BENCH_REPEATS)
print("PROBE QUALITY",final_probe)
print("PROBE PAIRED SPEED",probe_perf)
AUDIT_REF=get_reference(audit_items,AUDIT_DECISIONS)
assert {x["case"] for x in AUDIT_REF}.isdisjoint({x["case"] for x in PROBE_REF})
audit_result=quality(student,audit_items,AUDIT_REF)
audit_perf=benchmark_paired(teacher,student,repeats=max(15,BENCH_REPEATS))
print("UNTOUCHED AUDIT",audit_result)
print("GPU minimum speedup ratio",audit_perf["min_gpu_speedup"])
print("Wall minimum speedup ratio",audit_perf["min_wall_speedup"])


PROBE QUALITY {'decisions': 120, 'teacher_agreement': 0.9916666666666667, 'gold_accuracy': 0.8666666666666667, 'teacher_gold_accuracy': 0.875, 'gold_drop': 0.008333333333333333, 'mean_js': 0.000756681559929954}
PROBE PAIRED SPEED {'median_gpu_speedup': 1.0201556372728713, 'min_gpu_speedup': 1.0188054764390257, 'median_wall_speedup': 1.0202762952245243, 'min_wall_speedup': 1.0187397215745448, 'per_shape': [{'shape': 0, 'batch': 2, 'tokens': 315, 'native_gpu_ms': 53.597103118896484, 'candidate_gpu_ms': 52.46872138977051, 'gpu_speedup': 1.0215057981067168, 'native_wall_ms': 53.640556000004835, 'candidate_wall_ms': 52.49547900007201, 'wall_speedup': 1.0218128688745036, 'native_p95_ms': 54.66188011169434, 'candidate_p95_ms': 54.62674503326416, 'n': 20}, {'shape': 1, 'batch': 2, 'tokens': 391, 'native_gpu_ms': 67.85102462768555, 'candidate_gpu_ms': 66.59860610961914, 'gpu_speedup': 1.0188054764390257, 'native_wall_ms': 67.88745350013414, 'candidate_wall_ms': 66.63866350004355, 'wall_speedup'

In [14]:
#@title 11. Self-contained full checkpoint: save once, offline reload, verify parity
(DEST/"encoder").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizer").mkdir(parents=True,exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_v50.py").write_text(V50_RUNTIME_CODE,encoding="utf-8")
config=dict(
 format="laya-v50-all28-bruteforce-v1",
 model_type="laya-v50-selective-hybrid-svd",
 head_context_budget=64,head_layers=int(source_config["head_layers"]),
 source_model=TEACHER_REPO,source_revision=teacher_info.sha,
 compressed_projections=selected_specs,
 source_model_layers=28,
 candidate_families=len(families),
 ranks_tested=list(CANDIDATE_RANKS),
 selected_training_step=best_step,train_steps_requested=TRAIN_STEPS,
 selection_mode=selection_mode,
 output_scope="typed-decision option scores only")
(DEST/"config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")
state=student.state_dict()
assert not any("._shared." in k for k in state),(
    "Detected duplicate shared leader parameters in state_dict")
cpu_state=OrderedDict(
    (k,t.detach().cpu().contiguous().clone()) for k,t in state.items())
model_file=DEST/"model.safetensors"
save_file(cpu_state,str(model_file),metadata={
    "format":"pt","source_revision":str(teacher_info.sha),
    "architecture":"laya-v50-all28-bruteforce-v1"})
del cpu_state,state
gc.collect()
baseline_checkpoint=(parent_dir/"model.safetensors")
original_bytes=baseline_checkpoint.stat().st_size
compressed_bytes=model_file.stat().st_size
saved_bytes=original_bytes-compressed_bytes
assert saved_bytes>0,(original_bytes,compressed_bytes)
print("Original bytes:",original_bytes)
print("New standalone model bytes:",compressed_bytes)
print("ACTUAL DISK SAVING:",saved_bytes,"bytes",round(saved_bytes/1e6,3),"MB")
reload_model,reload_tok=v50.load_v50_model(DEST,device="cuda")
smoke=batch(probe_items,0,BATCH)
with torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    before,_=student(**smoke)
    after,_=reload_model(**smoke)
reload_diff=float((before.float()-after.float()).abs().max())
reload_choices=bool(torch.equal(before.argmax(-1),after.argmax(-1)))
reload_close=bool(torch.allclose(before.float(),after.float(),rtol=2e-3,atol=2e-3))
assert reload_choices and reload_close,(reload_diff,reload_choices,reload_close)
del reload_model
gc.collect();torch.cuda.empty_cache()
print("PASS: standalone offline reload matches original candidate",reload_diff)


Original bytes: 1684141004
New standalone model bytes: 1553546092
ACTUAL DISK SAVING: 130594912 bytes 130.595 MB
PASS: standalone offline reload matches original candidate 0.0


In [15]:
#@title 12. Strict gates, final reports, standalone model card and optional HF upload
gates=dict(
 probe_quality=quality_gate(final_probe),
 holdout_teacher_agreement=audit_result["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
 holdout_gold_drop=audit_result["gold_drop"]<=MAX_GOLD_DROP_PP/100.,
 holdout_mean_js=audit_result["mean_js"]<=MAX_JS,
 gpu_nonregression=audit_perf["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO,
 wall_nonregression=audit_perf["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO,
 minimum_20MB_storage_saved=saved_bytes>=MIN_SAVED_BYTES,
 local_reload_parity=reload_choices and reload_close)
gates["all_pass"]=all(gates.values())
report=dict(
 version="V5.0",
 architecture="all-28-layer attention+MLP own-SVD and selective shared-delta search",
 source_model=TEACHER_REPO,source_revision=teacher_info.sha,
 hardware=torch.cuda.get_device_name(0),pytorch=torch.__version__,
 tested_families=len(families),completed_individual_trials=total_trials,
 tested_ranks=list(CANDIDATE_RANKS),
 screen_decisions=SCREEN_DECISIONS,
 shortlisted_candidates=len(candidates),
 greedy_attempts=selection_attempts,
 selected_projections=selected_specs,
 selected_training_step=best_step,
 train_results=training_log,
 before_unique_parameters=native_unique,
 after_unique_parameters=unique_student,
 parameter_reduction=native_unique-unique_student,
 baseline_checkpoint_bytes=original_bytes,
 full_standalone_checkpoint_bytes=compressed_bytes,
 saved_checkpoint_bytes=saved_bytes,
 final_probe=final_probe,probe_paired_timings=probe_perf,
 final_holdout_audit=audit_result,final_holdout_timings=audit_perf,
 offline_max_abs_difference=reload_diff,
 gates=gates,
 limitations="PROBE is used for hundreds of comparisons; audit is a small single heldout set. Low-rank wrappers add computations and are not guaranteed to accelerate inference.")
(DEST/"v50_report.json").write_text(json.dumps(report,indent=2))
card=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- svd
- layerwise-compression
- inference-optimization
- standalone
---

# Laya V5.0 — All 28-layer brute-force hybrid SVD model

This model searches every attention QKV/output and feed-forward input/output
linear projection of **all 28 ModernBERT layers**, with additional
shared-delta projections for nine consecutive pairs of local-attention layers.

Only **{len(selected_specs)}** candidates were ultimately retained.
Every other layer uses its unchanged pretrained parameters.

- Original unique parameters: **{native_unique:,}**
- New unique parameters: **{unique_student:,}**
- Actual full FP32 checkpoint saving: **{saved_bytes:,} bytes**
- Probe teacher agreement: **{final_probe['teacher_agreement']:.3%}**
- Holdout teacher agreement: **{audit_result['teacher_agreement']:.3%}**
- Holdout gold accuracy: **{audit_result['gold_accuracy']:.3%}**
- Holdout JS divergence: **{audit_result['mean_js']:.7f}**
- GPU median speed ratio: **{audit_perf['median_gpu_speedup']:.3f}×**
- All validation gates passed: **{gates['all_pass']}**

Full model, tokenizer, encoder config, and the exact offline loader are
included. After downloading THIS repository once:

```python
from modeling_laya_v50 import load_v50_model
model,tokenizer=load_v50_model("/path/to/full/model",device="cuda")
answers=model.predict(tokenizer,dict(road_direction="left",speed=7),
    dict(steering=dict(type="choice",instructions="Steer",
         criteria=dict(left="left",straight="straight",right="right"))))
print(answers)
```

Fidelity measured on a narrow, one-time 200-decision holdout.
Storage compression does not automatically reduce GPU inference latency.
"""
(DEST/"README.md").write_text(card,encoding="utf-8")
(DEST/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
def directory_size(path):
    return sum(x.stat().st_size for x in Path(path).rglob("*") if x.is_file())
report["all_model_files_bytes"]=directory_size(DEST)
(DEST/"v50_report.json").write_text(json.dumps(report,indent=2))
print("PUBLICATION GATES",json.dumps(gates,indent=2))
print("Standalone checkpoint:",model_file)
print("Standalone directory bytes:",directory_size(DEST))
if not gates["all_pass"]:
    print("NOT VALIDATED: local experimental checkpoint saved, HF upload BLOCKED")
elif PUBLISH_TO_HF:
    notebook_login()
    uploader=HfApi()
    name=uploader.whoami()["name"]
    assert name=="vtava",f"Expected HF vtava, got {name!r}"
    assert STUDENT_REPO=="vtava/Laya-v50-all28-bruteforce"
    if uploader.repo_exists(STUDENT_REPO,repo_type="model"):
        existing=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
        if existing-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError("Model already contains files. Refusing overwrite.")
    else:
        created=uploader.create_repo(STUDENT_REPO,repo_type="model",
                                     private=PRIVATE_REPO,exist_ok=False)
        print("Created HF model:",str(created))
    uploader.upload_large_folder(
        repo_id=STUDENT_REPO,repo_type="model",folder_path=str(DEST))
    remote=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
    required={"model.safetensors","modeling_laya_v50.py","config.json",
              "encoder/config.json","README.md","v50_report.json"}
    assert required.issubset(remote),f"Missing remote files: {required-remote}"
    print("PUBLISHED:",f"https://huggingface.co/{STUDENT_REPO}")
else:
    print("Publication disabled. All local artifacts:",DEST)
print("Reference model unchanged:",TEACHER_REPO)


PUBLICATION GATES {
  "probe_quality": true,
  "holdout_teacher_agreement": false,
  "holdout_gold_drop": true,
  "holdout_mean_js": true,
  "gpu_nonregression": true,
  "wall_nonregression": true,
  "minimum_20MB_storage_saved": true,
  "local_reload_parity": true,
  "all_pass": false
}
Standalone checkpoint: /content/Laya_V50_All28_BruteForce/standalone_model/model.safetensors
Standalone directory bytes: 1557265503
NOT VALIDATED: local experimental checkpoint saved, HF upload BLOCKED
Reference model unchanged: vtava/Laya-1.2-speedup-standalone
